## TASK 2: Data Cleaning and Engineering

In [17]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, countDistinct

from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml import Pipeline

In [18]:
spark_session = (
    SparkSession.builder
    .appName("Task_2_session")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

In [19]:
parquet_dataset_directory = "data/raw/cifer-fraud-detection-AF/train"

fraud_detection_dataframe = spark_session.read.parquet(
    parquet_dataset_directory
)
print("DATASET LOADED SUCCESSFULLY!!!")

DATASET LOADED SUCCESSFULLY!!!


In [20]:
fraud_detection_dataframe.show(10, truncate=False)

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|type    |amount   |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|143 |PAYMENT |728.68   |C895984064 |157370.38    |296733.47     |M1248796614|946814.15     |21827.99      |0      |0             |
|83  |PAYMENT |37025.37 |C1166835933|12.44        |23912.77      |C266527134 |435103.83     |18206.87      |0      |0             |
|103 |CASH_IN |331069.91|C1589363125|82.16        |59517.13      |C222506215 |1383838.87    |1740585.08    |0      |0             |
|77  |PAYMENT |5097.55  |C109893045 |1272721.12   |1954746.32    |C250174919 |565635.26     |40351.13      |0      |0             |
|53  |CASH_IN |240031.01|C168215249 |1514711.38   |1464170.31    |C189987355

#### Information of the dataset

In [21]:
total_record_count = fraud_detection_dataframe.count()
total_column_count = len(fraud_detection_dataframe.columns)

print(f"Total Records : {total_record_count:,}")
print(f"Total Columns : {total_column_count}")


Total Records : 21,000,000
Total Columns : 11


In [22]:
partition_count = fraud_detection_dataframe.rdd.getNumPartitions()

print(f"Number of Spark Partitions : {partition_count}")

Number of Spark Partitions : 11


#### Missing Value Inspection

In [23]:
missing_value_summary = fraud_detection_dataframe.select([

    count(
        when(
            col(column_name).isNull(),
            column_name
        )
    ).alias(column_name)

    for column_name in fraud_detection_dataframe.columns

])

missing_value_summary.show(truncate=False)

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|0   |0   |0     |0       |0            |0             |0       |0             |0             |0      |0             |
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



There are no missing values. The data is consistent and clean. No imputation techniques needed here.

#### Checking for Duplicate Records

In [24]:
duplicate_record_count = (
    total_record_count -
    fraud_detection_dataframe.dropDuplicates().count()
)


In [25]:
print(f"Duplicate Records : {duplicate_record_count}")

Duplicate Records : 0


In [26]:
# Unique Values Per Feature
feature_cardinality = []

for feature_name in fraud_detection_dataframe.columns:

    unique_value_count = (
        fraud_detection_dataframe
        .select(countDistinct(feature_name))
        .first()[0]
    )

    feature_cardinality.append(

        (
            feature_name,
            unique_value_count
        )

    )

spark_session.createDataFrame(

    feature_cardinality,

    ["Feature", "UniqueValueCount"]

).toPandas().to_csv(

    "dashboard1_feature_cardinality.csv",

    index=False

)


# Fraud Distribution
fraud_detection_dataframe.groupBy(
    "isFraud"
).count().toPandas().to_csv(

    "dashboard1_fraud_distribution.csv",

    index=False

)


# Transaction Type Distribution
fraud_detection_dataframe.groupBy(
    "type"
).count().toPandas().to_csv(

    "dashboard1_transaction_type_distribution.csv",

    index=False

)

print("\nDashboard 1 export completed successfully.")


Dashboard 1 export completed successfully.


## Removing the Identifier Columns as they are redundant

In [27]:
prepared_fraud_dataframe = (

    fraud_detection_dataframe

    .drop(
        "step",
        "nameOrig",
        "nameDest"
    )

)

### Encoding Categorical Variables

In [28]:
transaction_type_indexer = StringIndexer(

    inputCol="type",

    outputCol="transactionTypeIndex",

    handleInvalid="keep"

)

In [29]:
numerical_feature_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

In [30]:
assembler_input_columns = (

    numerical_feature_columns +

    ["transactionTypeIndex"]

)

feature_vector_assembler = VectorAssembler(

    inputCols=assembler_input_columns,

    outputCol="assembledFeatures"

)

#### Feature Scaling

In [31]:
feature_scaler = StandardScaler(inputCol="assembledFeatures", outputCol="features", withMean=False, withStd=True)

In [32]:
training_dataframe, testing_dataframe = (prepared_fraud_dataframe.randomSplit([0.8, 0.2], seed=3))

In [33]:
print(training_dataframe.count())
print(testing_dataframe.count())

16799975
4200025


In [34]:
feature_engineering_pipeline = Pipeline(
    stages=[transaction_type_indexer, feature_vector_assembler, feature_scaler]
    )

In [35]:
pipeline_model = feature_engineering_pipeline.fit(training_dataframe)

In [36]:
processed_training_dataframe = pipeline_model.transform(training_dataframe)

In [37]:
processed_testing_dataframe = pipeline_model.transform(testing_dataframe)

#### Exporting the training and testing dataset

In [38]:
processed_training_output_directory = (
    "data/processed/training_dataset"
)

(
    processed_training_dataframe
    .write
    .mode("overwrite")
    .parquet(processed_training_output_directory)
)

print("Processed training dataset exported successfully.")

Processed training dataset exported successfully.


In [39]:
processed_testing_output_directory = (
    "data/processed/testing_dataset"
)

(
    processed_testing_dataframe
    .write
    .mode("overwrite")
    .parquet(processed_testing_output_directory)
)

print("Processed testing dataset exported successfully.")

Processed testing dataset exported successfully.


In [40]:
print("VERIFYING EXPORTED DATASETS")
print("=" * 80)

print("Training Dataset")

spark_session.read.parquet(
    processed_training_output_directory
).printSchema()

print("\nTesting Dataset")

spark_session.read.parquet(
    processed_testing_output_directory
).printSchema()

VERIFYING EXPORTED DATASETS
Training Dataset
root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)
 |-- transactionTypeIndex: double (nullable = true)
 |-- assembledFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)


Testing Dataset
root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)
 |-- transactionTypeIndex: double (nullable = true)
 |-- assembledFeatures: vector (nullable = true)
 |-- feat